# RetinexFormer ×8 + HVI-CIDNet fixed fusion

Select a GPU runtime. Place the original RetinexFormer weights and LOLv2 Real official Test pairs in Drive. Edit the paths below to match your Drive. To compare per-image results, supply the original 25/75 report ZIP or its per_image_official_test.csv. The notebook calls the same inference and evaluation scripts as the local workflow.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys
from zipfile import ZipFile
from google.colab import drive

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
retinex = drive_root / "RetinexFormer_weights/LOL_v2_real.pth"
test = drive_root / "datasets/LOLv2/Real_captured/Test"
reference = drive_root / "RetinexFormer_x8_HVI_FixedFusion_Official100_Report.zip"  # or a CSV, or None
run = drive_root / "fusion-runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
repo = Path("/content/myRetinexformer-fusion")
hvi = Path("/content/HVI-CIDNet-fusion")

def call(args, cwd=None):
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)

if not repo.exists():
    call(["git", "clone", "--single-branch", "--branch", "fusion-model",
          "https://github.com/Silly-Gooses/myRetinexformer.git", repo])
if not hvi.exists():
    call(["git", "clone", "--no-checkout", "https://github.com/Fediory/HVI-CIDNet.git", hvi])
call(["git", "checkout", "--detach", "eb43d7d91e9a336c66856824ff9e4603ae41f408"], cwd=hvi)
call([sys.executable, "-m", "pip", "install", "-r", repo / "requirements-fusion.txt"])
call(["git", "rev-parse", "HEAD"], cwd=repo)

In [ ]:
from huggingface_hub import hf_hub_download

weights = hf_hub_download("Fediory/HVI-CIDNet", "weights/LOLv2_real/best_PSNR.pth")
common = ["--retinex-checkpoint", retinex, "--hvi-checkpoint", weights,
          "--hvi-repo", hvi, "--device", "cuda"]
first = next(iter(sorted((test / "Low").glob("*.png"))))
call([sys.executable, "scripts/run_fusion.py", *common, "--input", first,
      "--compare", "--output", run / "smoke"], cwd=repo)

In [ ]:
reference_args = []
if reference is not None:
    csv_path = reference
    if reference.suffix.lower() == ".zip":
        csv_path = run / "reference.csv"
        with ZipFile(reference) as report:
            csv_path.write_bytes(report.read("per_image_official_test.csv"))
    reference_args = ["--reference-csv", csv_path]
call([sys.executable, "scripts/evaluate_fusion.py", *common, "--test-root", test,
      "--output", run / "official100", "--save-images", *reference_args], cwd=repo)
summary = json.loads((run / "official100/summary.json").read_text())
print(json.dumps(summary, indent=2))
print("Saved CSV, JSON and PNGs in", run)

Primary scores are mean per-image float RGB PSNR and author-style RGB8 SSIM. Ground truth is read after prediction. Earlier project experiments examined the Test set and informed the 75% HVI weight, so these results are exploratory.